# Experiment 1 — Naive Baseline

**Goal:** Get a working end-to-end pipeline as fast as possible.

**Approach:** Random Forest with minimal preprocessing and a single 80/20 holdout split.

**What this experiment established:** A complete runnable skeleton. Every subsequent experiment builds on this structure.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error# Load datasets
train = pd.read_csv("train_v9rqX0R.csv")
test = pd.read_csv("test_AbJTz2l.csv")

# Save test identifiers for submission
test_ids = test[['Item_Identifier','Outlet_Identifier']]# Combine datasets for preprocessing
train['source'] = 'train'
test['source'] = 'test'
test['Item_Outlet_Sales'] = 0

data = pd.concat([train, test], ignore_index=True)

# -------------------------
# DATA CLEANING
# -------------------------

# Fill missing Item_Weight with mean
data['Item_Weight'] = data['Item_Weight'].fillna(data['Item_Weight'].mean())

# Fill missing Outlet_Size with mode
data['Outlet_Size'] = data['Outlet_Size'].fillna(data['Outlet_Size'].mode()[0])

# Fix inconsistent Item_Fat_Content values
data['Item_Fat_Content'] = data['Item_Fat_Content'].replace({
    'LF':'Low Fat',
    'low fat':'Low Fat',
    'reg':'Regular'
})

# -------------------------
# FEATURE ENGINEERING
# -------------------------

# Create item category from Item_Identifier
data['Item_Category'] = data['Item_Identifier'].str[:2]

# Outlet age
data['Outlet_Age'] = 2026 - data['Outlet_Establishment_Year']

# -------------------------
# ENCODING
# -------------------------

cat_cols = [
    'Item_Fat_Content',
    'Item_Type',
    'Outlet_Size',
    'Outlet_Location_Type',
    'Outlet_Type',
    'Item_Category'
]

le = LabelEncoder()

for col in cat_cols:
    data[col] = le.fit_transform(data[col])

# Encode identifiers
data['Outlet_Identifier'] = le.fit_transform(data['Outlet_Identifier'])
data['Item_Identifier'] = le.fit_transform(data['Item_Identifier'])

# -------------------------
# SPLIT BACK TRAIN & TEST
# -------------------------

train = data[data['source']=='train'].drop(['source'], axis=1)
test = data[data['source']=='test'].drop(['source','Item_Outlet_Sales'], axis=1)

X = train.drop('Item_Outlet_Sales', axis=1)
y = train['Item_Outlet_Sales']

# Train validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------
# MODEL TRAINING
# -------------------------

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

# Validation score
pred_val = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, pred_val))

print("Validation RMSE:", rmse)

# -------------------------
# TRAIN ON FULL DATA
# -------------------------

model.fit(X, y)

# Predict test
test_predictions = model.predict(test)

# -------------------------
# CREATE SUBMISSION
# -------------------------

submission = pd.DataFrame({
    'Item_Identifier': test_ids['Item_Identifier'],
    'Outlet_Identifier': test_ids['Outlet_Identifier'],
    'Item_Outlet_Sales': test_predictions
})

submission.to_csv("submission/predictions1.csv", index=False)

print("Submission file created")

Validation RMSE: 1044.122975760547
Submission file created
